In [1]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, AzureCliCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition

load_dotenv()

# --- Setup clients ---

if os.getenv("USE_AZURE_CLI_CREDENTIALS", "false").lower() == "true":
    credential = AzureCliCredential()
else:
    credential = DefaultAzureCredential()
    
project_client = AIProjectClient(
    endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    credential=credential,
)
openai_client = project_client.get_openai_client()

# --- Step 1: Create an agent ---
agent = project_client.agents.create_version(
    agent_name="TravelBuddy",
    definition=PromptAgentDefinition(
        model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
        instructions=(
            "You are TravelBuddy, a friendly travel assistant. "
            "You help users plan trips, suggest destinations, and answer travel questions. "
            "Keep your answers short, concise, and easy to grasp."
        ),
    ),
)
print(f"Agent created — name: {agent.name}, version: {agent.version}")

# --- Step 2: Start a conversation ---
conversation = openai_client.conversations.create()
print(f"Conversation started — id: {conversation.id}")

# --- Step 3: First turn ---
user_input = "I have a week off in March. Suggest 3 warm destinations."
print(f"\n********\nUser: {user_input}")
response = openai_client.responses.create(
    conversation=conversation.id,
    input=user_input,
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
)
print(f"Agent: {response.output_text}")

# --- Step 4: Follow-up (multi-turn) ---
user_input = "Tell me more about the second option. What should I pack?"
print(f"\n********\nUser: {user_input}")
response = openai_client.responses.create(
    conversation=conversation.id,
    input=user_input,
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
)
print(f"Agent: {response.output_text}")

# --- Cleanup ---
openai_client.conversations.delete(conversation_id=conversation.id)
project_client.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
print("Cleaned up.")

Agent created — name: TravelBuddy, version: 1
Conversation started — id: conv_1b3711b42b6b6d0700ZdMubyxNqD2RbPxEDznev9wQHFsslhWh

********
User: I have a week off in March. Suggest 3 warm destinations.
Agent: - **Canary Islands (Spain)** — Warm, sunny, short flight from much of Europe; great beaches + hiking (Tenerife/Lanzarote).  
- **Dubai, UAE** — Hot and dry in March; mix of beaches, desert trips, and city attractions.  
- **Phuket or Krabi (Thailand)** — Peak warm season; beautiful islands, clear water, great for a relaxing week.

If you tell me your departure city and whether you want “mostly beach” vs “mix of sightseeing,” I can narrow it down.

********
User: Tell me more about the second option. What should I pack?
Agent: **Dubai in March (1 week): what it’s like**
- **Weather:** Warm and dry; typically **mid‑20s to low‑30s °C (75–90°F)**. Evenings can be a bit cooler with breezes.
- **Best for:** Beach days, desert safari, rooftop dining, shopping/malls, day trip to **Abu Dha

In [2]:
! python 01_basic_agent.py

Agent created — name: TravelBuddy, version: 1
Conversation started — id: conv_fffd97e5dc41cebc00KJoPbwY8GZESzyffZkSbtIZDhSASWRVH

********
User: I have a week off in March. Suggest 3 warm destinations.
Agent: - **Canary Islands, Spain (Tenerife/Gran Canaria)** – Reliably warm in March, beaches + hiking (Teide), easy resort downtime.  
- **Marrakech + Agafay Desert, Morocco** – Warm, sunny days, great food/markets; add a desert camp or day trip to the Atlas Mountains.  
- **Dubai, UAE** – Hot and dry in March, great for pools/beaches, desert safari, and easy day trips (Abu Dhabi).

********
User: Tell me more about the second option. What should I pack?
Agent: **Marrakech + Agafay Desert (7 days)**  
- **Weather in March:** Generally warm days and cooler evenings. Expect sun, but nights (and early mornings) can feel chilly—especially in the desert and if you head toward the Atlas Mountains.  
- **What to do:**  
  - 3–4 days in **Marrakech**: souks, Jardin Majorelle, Bahia Palace, hamma